In [1]:
import json
import csv
import os
from datetime import datetime
from pathlib import Path

# Setup paths
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

USERS_FILE   = DATA_DIR / "users.json"
EXPENSES_FILE = DATA_DIR / "expenses.csv"
BUDGETS_FILE  = DATA_DIR / "budgets.json"

EXPENSE_HEADERS = ["id", "username", "date", "category", "amount", "note"]
CATEGORIES = ["Food", "Transport", "University fees", "Shopping", "Other"]

def load_json(path, default):
    """Safely load JSON, return default if missing or corrupted."""
    if not path.exists():
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError):
        return default

def save_json(path, data):
    """Safely save to JSON."""
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        return True
    except (PermissionError, OSError) as e:
        print(f"[ERROR] Could not save {path}: {e}")
        return False

def load_expenses():
    """Read CSV safely, skipping bad rows."""
    if not EXPENSES_FILE.exists():
        return []
    rows = []
    try:
        with open(EXPENSES_FILE, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for r in reader:
                try:
                    r["amount"] = float(r["amount"])
                    rows.append(r)
                except (ValueError, TypeError):
                    continue # Skip corrupted rows gracefully
    except Exception as e:
        print(f"[ERROR] Reading expenses failed: {e}")
    return rows

def write_expenses(expenses):
    """Overwrite the entire CSV file (used for edits and deletes)."""
    try:
        with open(EXPENSES_FILE, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=EXPENSE_HEADERS)
            writer.writeheader()
            writer.writerows(expenses)
        return True
    except Exception as e:
        print(f"[ERROR] Writing expenses failed: {e}")
        return False

In [2]:
def user_login(username, password):
    """Authenticate user. Returns True if successful."""
    users = load_json(USERS_FILE, {})
    if username in users and users[username]["password"] == password:
        return True
    return False

def register_user(username, password):
    """Create a new user."""
    users = load_json(USERS_FILE, {})
    if username in users:
        return False
    users[username] = {"password": password}
    save_json(USERS_FILE, users)
    return True

def set_monthly_budget(username, amount):
    """Set the overall monthly budget for a user."""
    try:
        amount = float(amount)
        if amount <= 0: raise ValueError
    except ValueError:
        print("[ERROR] Budget must be a positive number.")
        return False
    
    budgets = load_json(BUDGETS_FILE, {})
    budgets[username] = amount
    save_json(BUDGETS_FILE, budgets)
    print(f"[OK] Monthly budget set to Rs. {amount:,.2f}")
    return True

In [7]:
def add_record(username, category, amount, note="", date=None):
    """Add expense and trigger budget alerts."""
    try:
        amount = float(amount)
        if amount <= 0: raise ValueError
    except (ValueError, TypeError):
        print("[ERROR] Invalid amount.")
        return False

    expenses = load_expenses()
    new_id = max([int(e["id"]) for e in expenses], default=0) + 1
    date_str = date if date else datetime.now().strftime("%Y-%m-%d")

    record = {
        "id": new_id, "username": username, "date": date_str,
        "category": category, "amount": amount, "note": note
    }
    
    expenses.append(record)
    write_expenses(expenses)
    print(f"[OK] Added {category} - Rs. {amount:.2f}")
    
    check_budget_alert(username)
    return True

def check_budget_alert(username):
    """Feature 4: Warn students when approaching/exceeding budget."""
    budgets = load_json(BUDGETS_FILE, {})
    if username not in budgets:
        return 
    
    limit = budgets[username]
    current_month = datetime.now().strftime("%Y-%m")
    expenses = [e for e in load_expenses() if e["username"] == username and e["date"].startswith(current_month)]
    total_spent = sum(e["amount"] for e in expenses)
    
    print("\n--- BUDGET ALERT ---")
    print(f"Spent: Rs. {total_spent:,.2f} out of Rs. {limit:,.2f}")
    
    # Check for exact match first
    if total_spent > limit:
        print("🚨 DANGER: You have EXCEEDED your monthly budget!")
    elif total_spent == limit:
        print("🛑 STOP: You have used exactly 100% of your budget. It is finished!")
    elif total_spent >= (limit * 0.8):
        print("⚠️ WARNING: You have used over 80% of your budget!")
    else:
        print(f"✅ ON TRACK: You have Rs. {limit - total_spent:,.2f} remaining.")
        
    print("---------------------\n")

def delete_record(username, record_id):
    """Delete a record by ID."""
    expenses = load_expenses()
    new_expenses = [e for e in expenses if not (e["username"] == username and str(e["id"]) == str(record_id))]
    if len(new_expenses) == len(expenses):
        return False
    write_expenses(new_expenses)
    return True

def edit_record(username, record_id, new_amount=None, new_category=None, new_note=None):
    """Feature 5: Edit an existing record."""
    expenses = load_expenses()
    for e in expenses:
        if e["username"] == username and str(e["id"]) == str(record_id):
            if new_amount: 
                try: e["amount"] = float(new_amount)
                except ValueError: print("[ERROR] Bad amount.")
            if new_category: e["category"] = new_category
            if new_note: e["note"] = new_note
            write_expenses(expenses)
            print("[OK] Record updated.")
            return True
    print("[ERROR] Record not found.")
    return False


In [8]:
def search_records(username, category=None, date=None):
    """Feature 5: Search by date or category."""
    results = [e for e in load_expenses() if e["username"] == username]
    if category and category != "All":
        results = [e for e in results if e["category"] == category]
    if date:
        results = [e for e in results if e["date"] == date]
    return results
    
def generate_insights(df, budget):
    insights = []
    if df.empty:
        return ["Add some transactions to see your personalized insights!"]
    cat_totals = df.groupby("category")["amount"].sum()
    top_cat = cat_totals.idxmax()
    top_amt = cat_totals.max()
    insights.append(f"🏆 **Top Category:** You spent the most on **{top_cat}** (Rs. {top_amt:,.2f}).")
    
    date_totals = df.groupby("date")["amount"].sum()
    top_date = date_totals.idxmax()
    top_date_amt = date_totals.max()
    insights.append(f"📅 **Highest Spending Day:** You spent **Rs. {top_date_amt:,.2f}** on **{top_date}**.")
    
    if budget > 0:
        total_spent = df["amount"].sum()
        # Fixed Logic for exact budget match
        if total_spent > budget:
            insights.append(f"🚨 **Budget Alert:** You have exceeded your budget by **Rs. {total_spent - budget:,.2f}**!")
        elif total_spent == budget:
            insights.append(f"🛑 **Budget Finished:** You have used exactly your full budget of Rs. {budget:,.2f}.")
        elif total_spent > (budget * 0.8):
            insights.append(f"⚠️ **Budget Warning:** You have used {((total_spent/budget)*100):.1f}% of your budget. Slow down!")
        else:
            insights.append(f"✅ **On Track:** You have Rs. {budget - total_spent:,.2f} left for the month. Keep it up!")
    return insights

def generate_report(username):
    """Feature 3: Expense Analysis (Totals, Highest category, Remaining Budget)."""
    current_month = datetime.now().strftime("%Y-%m")
    expenses = [e for e in load_expenses() if e["username"] == username and e["date"].startswith(current_month)]
    
    if not expenses:
        print("No expenses this month.")
        return

    # Calculate totals by category
    cat_totals = {}
    for e in expenses:
        cat_totals[e["category"]] = cat_totals.get(e["category"], 0) + e["amount"]
    
    total_spent = sum(cat_totals.values())
    highest_cat = max(cat_totals, key=cat_totals.get) if cat_totals else "None"
    
    # Get Budget
    budgets = load_json(BUDGETS_FILE, {})
    budget = budgets.get(username, 0)
    remaining = budget - total_spent if budget > 0 else 0
    
    # Print Report
    print(f"\n{'='*45}")
    print(f" EXPENSE REPORT: {username.upper()} ({current_month})")
    print(f"{'='*45}")
    for cat, amt in sorted(cat_totals.items(), key=lambda x: -x[1]):
        print(f" {cat:<18} Rs. {amt:>9,.2f}")
    print(f"{'-'*45}")
    print(f" TOTAL SPENT        Rs. {total_spent:>9,.2f}")
    if budget > 0:
        print(f" REMAINING BUDGET   Rs. {remaining:>9,.2f}")
        percent = (total_spent / budget) * 100
        print(f" BUDGET USED        {percent:.1f}%")
    print(f" HIGHEST CATEGORY   {highest_cat}")
    print(f"{'='*45}\n")


In [5]:
def run_terminal_portal():
    print("--- LOGIN ---")
    user = input("Username: ")
    pwd = input("Password: ")
    
    if not user_login(user, pwd):
        if input("Register? (y/n): ").lower() == "y":
            register_user(user, pwd)
            print("Registered! Please log in again.")
            return
        else:
            return

    while True:
        print("\n1. Set Budget\n2. Add Expense\n3. Search History\n4. Generate Report\n5. Exit")
        choice = input("Choice: ")
        
        if choice == "1":
            amt = input("Enter total monthly budget: ")
            set_monthly_budget(user, amt)
        elif choice == "2":
            print("Categories:", CATEGORIES)
            cat = input("Category: ")
            amt = input("Amount: ")
            note = input("Note: ")
            add_record(user, cat, amt, note)
        elif choice == "3":
            cat = input("Filter by category (or press enter): ")
            dt = input("Filter by date (YYYY-MM-DD, or press enter): ")
            results = search_records(user, cat or None, dt or None)
            for r in results: # FOR LOOP requirement
                print(f"ID:{r['id']} | {r['date']} | {r['category']} | Rs.{r['amount']} | {r['note']}")
        elif choice == "4":
            generate_report(user)
        elif choice == "5":
            break
